# DQPT in the Ising model

In this tutorial we will try to reproduce the results from
[this paper](https://arxiv.org/pdf/1206.2505.pdf). The needed packages are

In [ ]:
using MPSKit, TensorKit
# Model constructors for the MPSKit documentation and examples.
# Adapted from MPSKitModels.jl v0.4.7, src/models/{hamiltonians,transfermatrices}.jl:
# https://github.com/QuantumKitHub/MPSKitModels.jl/tree/v0.4.7/src/models
# Operators come from TensorKitTensors; chain assembly uses MPSKit directly.
#
# MIT License
# Copyright (c) 2021 Maarten Van Damme
#
# Permission is hereby granted, free of charge, to any person obtaining a copy
# of this software and associated documentation files (the "Software"), to deal
# in the Software without restriction, including without limitation the rights
# to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
# copies of the Software, and to permit persons to whom the Software is
# furnished to do so, subject to the following conditions:
#
# The above copyright notice and this permission notice shall be included in all
# copies or substantial portions of the Software.
#
# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
# OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
# SOFTWARE.

module ExampleModels

using MPSKit, TensorKit
using TensorKitTensors: SpinOperators, BosonOperators, HubbardOperators

export chain_hamiltonian, transverse_field_ising, heisenberg_XXX
export hubbard_model, bose_hubbard_model, hard_hexagon

# L specifies an open finite chain; otherwise unitcell specifies an infinite chain.
function chain_hamiltonian(twosite, onesite = nothing; L = nothing, unitcell::Integer = 1)
    nsites = isnothing(L) ? unitcell : L
    nsites isa Integer && nsites > 0 || throw(ArgumentError("chain length must be a positive integer"))
    nbonds = isnothing(L) ? nsites : nsites - 1
    terms = Pair[(i, i + 1) => twosite for i in 1:nbonds]
    if !isnothing(onesite)
        append!(terms, [i => onesite for i in 1:nsites])
    end
    spaces = fill(space(twosite, 1), nsites)
    constructor = isnothing(L) ? InfiniteMPOHamiltonian : FiniteMPOHamiltonian
    return constructor(spaces, terms)
end

# H = -J sum(σz_i σz_{i+1} + g σx_i).
function transverse_field_ising(
        T::Type{<:Number} = ComplexF64, symmetry::Type{<:Sector} = Trivial;
        J = 1.0, g = 1.0, kwargs...
    )
    ZZ = 4 * SpinOperators.S_z_S_z(T, symmetry; spin = 1 // 2)
    X = SpinOperators.σˣ(T, symmetry)
    return chain_hamiltonian(-J * ZZ, -J * g * X; kwargs...)
end
transverse_field_ising(symmetry::Type{<:Sector}; kwargs...) =
    transverse_field_ising(ComplexF64, symmetry; kwargs...)

# H = J sum(S_i ⋅ S_{i+1}); preserve the examples' spin-1 default.
function heisenberg_XXX(
        T::Type{<:Number} = ComplexF64, symmetry::Type{<:Sector} = Trivial;
        J = 1.0, spin = 1, kwargs...
    )
    term = J * SpinOperators.S_exchange(T, symmetry; spin)
    return chain_hamiltonian(term; kwargs...)
end
heisenberg_XXX(symmetry::Type{<:Sector}; kwargs...) =
    heisenberg_XXX(ComplexF64, symmetry; kwargs...)

# H = -t sum(e⁺_i e⁻_{i+1} + h.c.) + U sum(n↑ n↓) - mu sum(n).
function hubbard_model(
        T::Type{<:Number} = ComplexF64, particle_symmetry::Type{<:Sector} = Trivial,
        spin_symmetry::Type{<:Sector} = Trivial;
        t = 1.0, U = 1.0, mu = 0.0, kwargs...
    )
    # e_hopping includes the fermionic reordering sign and is Hermitian.
    hopping = HubbardOperators.e_hopping(T, particle_symmetry, spin_symmetry)
    interaction = HubbardOperators.ud_num(T, particle_symmetry, spin_symmetry)
    N = HubbardOperators.e_num(T, particle_symmetry, spin_symmetry)
    return chain_hamiltonian(-t * hopping, U * interaction - mu * N; kwargs...)
end

# H = -t sum(b⁺_i b⁻_{i+1} + h.c.) + U/2 sum(n(n-1)) - mu sum(n).
function bose_hubbard_model(
        T::Type{<:Number} = ComplexF64, symmetry::Type{<:Sector} = Trivial;
        cutoff::Integer = 5, t = 1.0, U = 1.0, mu = 0.0, kwargs...
    )
    hopping = BosonOperators.b_plus_b_min(T, symmetry; cutoff) +
        BosonOperators.b_min_b_plus(T, symmetry; cutoff)
    N = BosonOperators.b_num(T, symmetry; cutoff)
    interaction = N * (N - id(domain(N)))
    return chain_hamiltonian(-t * hopping, U / 2 * interaction - mu * N; kwargs...)
end

function hard_hexagon(T::Type{<:Number} = ComplexF64)
    P = Vect[FibonacciAnyon](:τ => 1)
    O = ones(T, P ⊗ P ← P ⊗ P)
    block(O, FibonacciAnyon(:I)) .*= 0
    return InfiniteMPO([O])
end

end

using .ExampleModels

Dynamical quantum phase transitions (DQPT in short) are signatures of equilibrium phase transitions in a dynamical quantity - the Loschmidt echo.
This quantity is given by $L(t) = \frac{-2}{N} ln(| < \psi(t) | \psi(0) > |) $ where $N$ is the system size.
One typically starts from a ground state and then quenches the Hamiltonian to a different point.
Non analycities in the Loschmidt echo are called 'dynamical quantum phase transitions'.

In the mentioned paper they work with

$H(g) = - \sum^{N-1}_{i=1} \sigma^z_i \sigma^z_{i+1} + g \sum_{i=1}^N \sigma^x_i$

and show that divergences occur when quenching across the critical point (g₀ → g₁) for $t^*_n = t^*(n+\frac{1}{2})$ with $t^* = \pi/e(g_1,k^*)$, $cos(k^*) = (1+g_0 g_1) / (g_0 + g_1)$, $ e(g,k) = \sqrt{(g-cos k)^2 + sin^2 k}$.

The outline of the tutorial is as follows. We will pick $g₀ = 0.5$, $g₁ = 2.0$, and perform the time evolution at different system sizes and compare with the thermodynamic limit.
For those $g$ we expect non-analyticities to occur at $t_n ≈ 2.35 (n + 1/2)$.

First we construct the Hamiltonian in MPO form, and obtain the pre-quenched ground state:

In [ ]:
L = 20
H₀ = transverse_field_ising(; L, g = -0.5)
ψ₀ = FiniteMPS(L, ℂ^2, ℂ^10)
ψ₀, _ = find_groundstate(ψ₀, H₀, DMRG());

## Finite MPS quenching

We can define a helper function that measures the Loschmidt echo:

In [ ]:
echo(ψ₀::FiniteMPS, ψₜ::FiniteMPS) = -2 * log(abs(dot(ψ₀, ψₜ))) / length(ψ₀)
@assert isapprox(echo(ψ₀, ψ₀), 0, atol = 1.0e-10)

We will initially use a two-site TDVP scheme to dynamically increase the bond dimension while time evolving, and later on switch to a faster one-site scheme. A single timestep can be done using

In [ ]:
H₁ = transverse_field_ising(; L, g = -2.0)
ψₜ = deepcopy(ψ₀)
dt = 0.01
ψₜ, envs, info = timestep(ψₜ, H₁, 0, dt, TDVP2(; trunc = truncrank(20)));

"envs" is a kind of cache object that keeps track of all environments in `ψ`. It is often advantageous to re-use the environment, so that MPSKit doesn't need to recalculate everything.

Putting it all together, we get

In [ ]:
function finite_sim(L; dt = 0.05, finaltime = 5.0)
    ψ₀ = FiniteMPS(L, ℂ^2, ℂ^10)
    H₀ = transverse_field_ising(; L, g = -0.5)
    ψ₀, _ = find_groundstate(ψ₀, H₀, DMRG())

    H₁ = transverse_field_ising(; L, g = -2.0)
    ψₜ = deepcopy(ψ₀)
    envs = environments(ψₜ, H₁, ψₜ)

    echos = [echo(ψₜ, ψ₀)]
    times = collect(0:dt:finaltime)

    for t in times[2:end]
        alg = t > 3 * dt ? TDVP() : TDVP2(; trunc = truncrank(50))
        ψₜ, envs, info = timestep(ψₜ, H₁, 0, dt, alg, envs)
        push!(echos, echo(ψₜ, ψ₀))
    end

    return times, echos
end

![](finite_timeev.png)

## Infinite MPS quenching

Similarly we could start with an initial infinite state and find the pre-quench ground state:

In [ ]:
ψ₀ = InfiniteMPS([ℂ^2], [ℂ^10])
H₀ = transverse_field_ising(; g = -0.5)
ψ₀, _ = find_groundstate(ψ₀, H₀, VUMPS());

The dot product of two infinite matrix product states scales as  $\alpha ^N$ where $α$ is the dominant eigenvalue of the transfer matrix.
It is this $α$ that is returned when calling

In [ ]:
dot(ψ₀, ψ₀)

so the Loschmidt echo takes on the pleasant form

In [ ]:
echo(ψ₀::InfiniteMPS, ψₜ::InfiniteMPS) = -2 * log(abs(dot(ψ₀, ψₜ)))
@assert isapprox(echo(ψ₀, ψ₀), 0, atol = 1.0e-10)

We make use of the `changebonds` machinery to grow the bond dimension. This can also be achieved through a two-site scheme.
Multiple algorithms are available, but we will only focus on `OptimalExpand()`.
Growing the bond dimension by $5$ can be done by calling:

In [ ]:
ψₜ = deepcopy(ψ₀)
H₁ = transverse_field_ising(; g = -2.0)
ψₜ, envs = changebonds(ψₜ, H₁, OptimalExpand(; trunc = truncrank(5)));

a single timestep is easy

In [ ]:
dt = 0.01
ψₜ, envs, info = timestep(ψₜ, H₁, 0, dt, TDVP(), envs);

With performance in mind we should once again try to re-use these "envs" cache objects.
The final code is

In [ ]:
function infinite_sim(dt = 0.05, finaltime = 5.0)
    ψ₀ = InfiniteMPS([ℂ^2], [ℂ^10])
    ψ₀, _ = find_groundstate(ψ₀, H₀, VUMPS())

    ψₜ = deepcopy(ψ₀)
    envs = environments(ψₜ, H₁, ψₜ)

    echos = [echo(ψₜ, ψ₀)]
    times = collect(0:dt:finaltime)

    for t in times[2:end]
        if t < 50dt # if t is sufficiently small, we increase the bond dimension
            ψₜ, envs = changebonds(ψₜ, H₁, OptimalExpand(; trunc = truncrank(1)), envs)
        end
        ψₜ, envs, info = timestep(ψₜ, H₁, 0, dt, TDVP(), envs)
        push!(echos, echo(ψₜ, ψ₀))
    end

    return times, echos
end

![](infinite_timeev.png)

---

*This notebook was generated using [Literate.jl](https://github.com/fredrikekre/Literate.jl).*